# 📓 Semana 4 · Dia 1 — Delta Lake: ACID, Time Travel e o _delta_log

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (Delta Lake) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Time Travel exercitado com versões da tabela |

---


## 📖 Teoria — O que o Delta Lake resolve

Parquet puro é um data lake sem transações: escrita interrompida corrompe, leitores veem estado inconsistente, updates são 'reescrever tudo'.

O **Delta Lake** adiciona ao Parquet:
- **ACID**: transações atômicas, consistentes, isoladas e duráveis
- **Time Travel**: consultar qualquer versão histórica
- **Schema enforcement e evolution**
- **MERGE/UPSERT** e **Change Data Feed**

Tecnicamente: um diretório com Parquet (dados) + `_delta_log/` (log JSON de transações). O log é a 'fonte da verdade'.


## 📖 Teoria — O _delta_log

Cada escrita (commit) adiciona um arquivo JSON `00000N.json` no `_delta_log`, descrevendo o que mudou (novos Parquet, arquivos removidos, metadados, constraints). Isso permite: leitura consistente, Time Travel e rollback.


### 💻 Na prática — Time Travel

Faça commits na tabela e viaje no tempo.


In [ ]:
# Garantir a tabela Bronze e um histórico de versões
df = spark.table("workspace.bronze.vendas_bronze")
print("Versões desta tabela:")
display(spark.sql("DESCRIBE HISTORY workspace.bronze.vendas_bronze").select("version", "timestamp", "operation").limit(5))

In [ ]:
# Criar alterações para ter várias versões
spark.sql("UPDATE workspace.bronze.vendas_bronze SET Country = 'BRASIL' WHERE Country = 'Brazil'")
spark.sql("UPDATE workspace.bronze.vendas_bronze SET Country = 'Brazil' WHERE Country = 'BRASIL'")
print("Foram criadas 2 novas versões (update).")

In [ ]:
# Time Travel: consultar uma versão antiga
# 1) Por versão
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("workspace.bronze.vendas_bronze")
# 2) Por timestamp
df_ts = spark.read.format("delta")\
    .option("timestampAsOf", "2024-01-01")\
    .table("workspace.bronze.vendas_bronze")
print("Versão 0:", df_v0.count(), "| Antes de 2024:", df_ts.count())

### 💻 Na prática — Restore e VACUUM

`RESTORE` volta a tabela para uma versão; `VACUUM` apaga arquivos antigos (destrói Time Travel antigo).


In [ ]:
%sql
-- Restore para a versão 0 (desfaz updates)
RESTORE TABLE workspace.bronze.vendas_bronze TO VERSION AS OF 0;
SELECT COUNT(*) FROM workspace.bronze.vendas_bronze;

In [ ]:
# VACUUM com retenção padrão (7 dias) — em tabelas grandes, libera espaço
# ATENÇÃO: torna irreversível o Time Travel para além da retenção
spark.sql("VACUUM workspace.bronze.vendas_bronze")  # opcional; pode demorar
print("VACUUM remove arquivos órfãos/antigos além da retenção.")

> 🎯 **Dica de prova**: Time Travel: `DESCRIBE HISTORY`, `versionAsOf`, `timestampAsOf`, `RESTORE TABLE ... TO VERSION`. VACUUM destrói Time Travel antigo — pergunta clássica de prova.


## 🎯 Exercícios de fixação

**1.** Quantas versões sua tabela tem agora? Use DESCRIBE HISTORY.

**2.** O que acontece com leitores ativos durante um VACUUM?

**3.** Crie uma tabela Delta nova e faça: insert, update, delete, e restaure para antes do delete.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Versões

`DESCRIBE HISTORY workspace.bronze.vendas_bronze` mostra a lista; a versão atual é a última linha.

**2.** VACUUM com leitores

VACUUM respeita a retenção (7 dias por padrão) e as transações ativas: arquivos ainda necessários não são removidos. Leitores ativos usam a versão que começaram a ler.

**3.** Tabela e restore

```sql
CREATE TABLE workspace.bronze.t_teste (id INT, v STRING) USING DELTA;
INSERT ...; UPDATE ...; DELETE ...;
RESTORE TABLE workspace.bronze.t_teste TO VERSION AS OF 1;
```



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*